# Acquirium client reference

This is the reference guide for the acquirium client: the query interface feature by feature, with the internals (`to_sparql`, text resolution) shown along the way. If you are new, start with `quickstart.ipynb`.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `uv run acquirium server --config deployments/WATERTAP/models/seawater-ro/acquirium.toml`
2. Connect:

In [ ]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`entity()` accepts a URI or a natural-language string. The alias defaults to what you typed; `alias=` or `.alias()` renames it.

In [ ]:
q = acq.query().entity("Pump")
q.metadata()

### How strings become URIs
Every string is resolved server-side by an embedding matcher. `resolve()` shows what a string resolves to — check it when a query returns something unexpected, and pass exact URIs when correctness matters (the top match is not always the intended one):

In [ ]:
acq.client.resolve("salt", kind="class", top_k=3)

## Follow relationships
`related()` adds a neighbour. By default it walks any non-hidden predicate up to 3 hops and keeps the nearest match. `via=` restricts traversal to one predicate or a list of them, `max_depth=` sets the reach, and `direction="upstream"/"downstream"` walks the S223 piping topology instead. Strings and URIs are both accepted.

In [ ]:
q = (
    acq.query().entity("Pump")
    .related("tank")
)
q.metadata()


## Attach data nodes
`measurement()` adds the data-bearing points of the current node: its own properties plus the ones hanging off its connection points. On an empty query it matches every registered stream in the plant.

In [ ]:
q = acq.query().entity("Pump").measurement()
q.metadata()

In [ ]:
acq.query().measurement().metadata()

## Filter data nodes
`where()` filters the node the pointer is on. Strings are resolved via the text matcher. The same keywords work inline on `entity()`, `related()` and `measurement()`.

In [ ]:
q = acq.query().measurement().where(quantity_kind="Pressure")
q.metadata()

In [ ]:
q = acq.query().measurement().where(unit="kg/s")
q.metadata()

In [ ]:
q = acq.query().measurement().where(unit="kg/s",substance="constituent salt")
q.metadata()

`unit`, `substance`, `quantity_kind` and `medium` are attributes, not methods: one vocabulary shared by `where()`, `include()`, `options()` and the inline keywords. `medium` covers both predicates that carry it — `s223:ofMedium` on a property and `s223:hasMedium` on a connection point — so one filter finds the brine points wherever the medium is declared. Wrap a value in `Not()` to exclude it instead.

In [ ]:
q_brine = acq.query().measurement().where(medium="brine")
q_brine.metadata()

## Inspect the query
Every query compiles to SPARQL against the server's graph — nothing is hidden. `to_sparql()` returns the compiled query without running it (check it when a result surprises you), and `metadata()` returns the full result as a polars DataFrame (`include_internals=True` keeps the ref and unit columns).

In [ ]:
q = acq.query().measurement().where(unit="kg/s", substance="constituent salt")

In [ ]:
print(q.to_sparql())

In [ ]:
df_meta = q.metadata()
df_meta

## Pull timeseries
`dataframe()` returns a polars frame. `shape="wide"` (the default) puts each data node in its own column, `"narrow"` is one row per reading. `Query.dataframe()` and `DataObject.dataframe()` take the same parameters with the same defaults, so `q.dataframe(...)` equals `q.data(...).dataframe(...)`.

In [ ]:
q = acq.query().measurement().where(unit="kg/s",substance="constituent salt")


end = datetime.now(tz=timezone.utc)
start = end - timedelta(hours=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

In [ ]:
q.dataframe(limit=1, order='desc', shape="wide")

## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [ ]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe()

## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [ ]:
data.units()

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [ ]:
data = data.convert_to("kg/min")
data.dataframe().head()

### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [ ]:
q = acq.query().entity(cls="system")
q.metadata()

The systems are hierarchically organized as:

In [ ]:
q = acq.query().entity(cls="system").related("system").alias("subsystem")
q.metadata()

We can see how many equipment we have in each system:

In [ ]:
q = acq.query().entity(cls="System").related("Equipment")
q.metadata()

In [ ]:
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

These are the equipment directly a member of each system.

`via="has member"` with `nearest=False` follows membership all the way down, so equipment nested in a subsystem counts too:

In [ ]:
q = acq.query().entity(cls="System").related("Equipment",via="has member",nearest=False)
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

Let's find the pumps in a specific system:


In [ ]:
q= acq.query().entity(uri='wbs:pretreatment-system').alias("system").related("pump")
q.metadata()

Let's find all the pumps and their data

In [ ]:
q = acq.query().entity("pump").measurement()
q.metadata().head()

Let's find all the data generating entites within a system:

In [ ]:
q = (acq.query().entity(uri = 'wbs:pretreatment-system').drop()
     .related('equipment').measurement(alias="sensor").include("unit"))
q.metadata()